# 06.17_DNB_analysis_R

DNB 模块与候选 OG 分析。

- 当前文件：`analysis/06_single_cell_analysis/06.17_DNB_analysis_R.ipynb`
- 原始来源：`Codes/06.16_R_DNB_analysis.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`DNBr`, `Seurat`, `cowplot`, `dplyr`, `ggplot2`, `ggridges`, `htmlwidgets`, `plotly`, `tidyr`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

### Demo

In [ ]:
library(Seurat)
library(DNBr)
library(dplyr)
library(ggplot2)

In [ ]:
# 加载数据
integrated_rpca <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/integrated_rpca.rds")

In [ ]:
table(integrated_rpca$species, integrated_rpca$Phylum)
table(integrated_rpca$BroadType)

In [ ]:
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/DNB_analysis"

In [ ]:
# 1. 准备数据
neural_obj <- subset(integrated_rpca, BroadType == "Neural")
phylum_orders <- c("Porifera", "Placozoa", "Cnidaria", "Bilateria")
neural_obj$Phylum <- factor(neural_obj$Phylum, levels = phylum_orders)

In [ ]:
table(neural_obj$species, neural_obj$Phylum)

In [ ]:
# Demo下采样，如果想跑全量数据，把下面这段注释掉即可
set.seed(42)
demo_indices <- unlist(lapply(phylum_orders, function(p){
    cells_in_p <- which(neural_obj$Phylum == p)
    sample(cells_in_p, min(length(cells_in_p), 3000)) # 每个门抽1000个
}))
neural_working <- neural_obj[, demo_indices]

In [ ]:
table(neural_working$species, neural_working$Phylum)

In [ ]:
neural_working$Clade <- ifelse(
  neural_working$Phylum %in% c("Porifera", "Placozoa"), 
  "Non-neural-clade", 
  "Neural-clade"
)
table(neural_working$species, neural_working$Clade)

In [ ]:
saveRDS(neural_working, paste0(out_dir, "/neural_working.rds"))

In [ ]:
# 2. 准备矩阵
dnb_data <- as.matrix(GetAssayData(neural_working, assay = "integrated", slot = "scale.data"))
dnb_meta <- neural_working$Phylum
names(dnb_meta) <- colnames(neural_working)

In [ ]:
# 3. 执行 DNB 计算 (核心步骤)
a_phylum <- DNBcompute(
    data = dnb_data, 
    meta = dnb_meta,
    # high_cutoff = -1  # 关键：跳过筛选，保留所有整合后的 OG
)

# 【及时保存 1】核心计算结果
saveRDS(a_phylum, paste0(out_dir, "/a_phylum_DNB_compute_results.rds"))

In [ ]:
# 4. 过滤结果
b_phylum <- DNBfilter(
    DNB_output = a_phylum, 
    ntop = 10,
    force_allgene = TRUE # 关键：确保模块背景一致性
)

# 【及时保存 2】过滤后的绘图对象
saveRDS(b_phylum, paste0(out_dir, "/b_phylum_DNB_filtered_obj.rds"))

In [ ]:
# 5. 提取并过滤结果
res_all <- resultAllExtract(
    object = b_phylum,
    group = "Placozoa", # 这里即便写了 Placozoa，返回的可能也是全量
    slot = "result"
)

# 【核心修正】只锁定在 Placozoa 阶段识别出的 10 个模块
res_placozoa_only <- res_all %>%
    filter(grepl("Placozoa", resource)) %>%
    # 按照组内局部排名 (1-10) 排序，这样第一行就是 Placozoa 的最强预警信号
    arrange(rank)

# 保存真正的扁盘动物起源模块表
write.csv(res_placozoa_only, paste0(out_dir, "/DNB_Origin_Modules_Placozoa_Clean.csv"), row.names = FALSE)

# 打印检查：现在你应该只看到 Placozoa_XX 相关的 ID 了
head(res_placozoa_only)

In [ ]:
# 6. 可视化
phylum_orders <- c("Porifera", "Placozoa", "Cnidaria", "Bilateria")

# 建议这里观察一下 Placozoa 还是 Cnidaria 出现了峰值
pdf(paste0(out_dir, "/DNB_Evolution_Phylum_Trend.pdf"), width = 8, height = 6)
# DNBplot(
#     b_phylum, 
#     ranking = 1, 
#     group = "Placozoa", 
#     show = TRUE, 
#     save_pdf = FALSE, 
#     meta_levels = phylum_orders
# )
# 指定resource
DNBplot(
    b_phylum, 
    resource = 'Placozoa_74', 
    show = TRUE, 
    save_pdf = FALSE, 
    meta_levels = phylum_orders
)
dev.off()

In [ ]:
library(cowplot)

# 定制化高亮绘图函数 (pp)
pp_custom <- function(df, y, meta_levels, red_node_index = 2) {
    df_plot <- df[, c("Names", y)]
    colnames(df_plot)[2] <- 'y'
    df_plot$Names <- factor(df_plot$Names, levels = meta_levels)
    
    ggplot(df_plot, aes(x = Names, y = y, group = 1)) +
        geom_point(size = 3) +
        geom_line(size = 1) +
        # 在指定节点画红点（默认为第2个节点：Placozoa）
        geom_point(data = df_plot[red_node_index, ], aes(x = Names, y = y), colour = "red", size = 4) +
        theme_classic(base_size = 15) +
        ggtitle(y) +
        theme(axis.text.x = element_text(face = "bold", color = "black", angle = 45, hjust = 1),
              plot.title = element_text(hjust = 0.5),
              legend.position = "none")
}

In [ ]:
# 针对最强信号 (Rank 1) 绘制四联精美图
# 假设 Rank 1 在 Placozoa 的 resource ID 为 "Placozoa_74" (请根据你实际运行结果修改)
# 如果不确定 ID，可以直接用 ranking 参数提取数据
df_pt <- ScoreExtract(
    b_phylum, 
    resource = 'Placozoa_74', 
    # ranking = 1, 
    # group = "Placozoa"
)

p1 <- pp_custom(df_pt, 'SCORE',   phylum_orders, red_node_index = 3)
p2 <- pp_custom(df_pt, 'PCC_IN',  phylum_orders, red_node_index = 3)
p3 <- pp_custom(df_pt, 'PCC_OUT', phylum_orders, red_node_index = 3)
p4 <- pp_custom(df_pt, 'SD',      phylum_orders, red_node_index = 3)

p_final <- plot_grid(p1, p2, p3, p4, ncol = 2)
ggsave(p_final, filename = paste0(out_dir, "/DNB_Highlighted_Rank1_Placozoa.pdf"), width = 10, height = 8)

message("所有分析结果已保存至：", out_dir)

### Top 10 Gene Modules 3D

In [ ]:
res_placozoa_only

In [ ]:
library(ggridges)

# 1. 批量抓取 res_placozoa_only 中这 10 个模块的完整演化轨迹
# 我们直接利用提取出的 resource 唯一标识符
placozoa_resources <- unique(res_placozoa_only$resource)

plot_list <- lapply(placozoa_resources, function(res_id){
  # 使用 resource 参数精准提取
  df <- ScoreExtract(b_phylum, resource = res_id)
  df$Module_ID <- res_id
  return(df)
})

plot_df <- do.call(rbind, plot_list)
plot_df

In [ ]:
library(plotly)
library(dplyr)
library(tidyr)

# ================= 1. 严格顺序的数据准备 =================

# 1.1 获取 plot_df 中已有的 Module 顺序（这是你根据 rank 排序后的顺序）
ordered_modules <- unique(plot_df$Module_ID) 
phylum_orders <- c("Bilateria", "Cnidaria", "Placozoa", "Porifera")

# 1.2 数据转换与插值预处理
df_wide <- plot_df %>%
  select(Names, Module_ID, SCORE) %>%
  # 关键：强制 Module_ID 遵循当前的先后顺序，不按字母重排
  mutate(Module_ID = factor(Module_ID, levels = ordered_modules)) %>%
  mutate(Names = factor(Names, levels = phylum_orders)) %>%
  # 展开为矩阵格式
  pivot_wider(names_from = Names, values_from = SCORE) %>%
  # 按照刚才设定的 factor 顺序排序行
  arrange(Module_ID) %>%
  as.data.frame()

# 提取数值矩阵
z_matrix <- as.matrix(df_wide[, -1])
row.names(z_matrix) <- df_wide$Module_ID

# --- 2. X 轴（时间）与 Y 轴（模块）双向平滑 ---
# 时间轴插值 (4 -> 50 点)
x_orig <- 1:4
x_interp <- seq(1, 4, length.out = 50)

# 模块轴插值 (10 -> 100 点) - 这样 Y 轴方向也会变得圆滑，像连绵的山脉
y_orig <- 1:nrow(z_matrix)
y_interp <- seq(1, nrow(z_matrix), length.out = 100)

# 执行双三次插值（使用 akima 包或简单的 apply 组合）
# 这里我们对每一行先做 X 轴平滑
z_temp <- t(apply(z_matrix, 1, function(y) spline(x_orig, y, xout = x_interp)$y))

# 再对每一列做 Y 轴平滑（让模块间过渡也圆滑）
z_final <- apply(z_temp, 2, function(x) spline(y_orig, x, xout = y_interp)$y)

# ================= 3. 绘制 3D 山脉地形图 =================

p_3d_final <- plot_ly(
    x = ~x_interp, 
    y = ~y_interp, 
    z = ~z_final,
    type = "surface",
    # 颜色配置：使用类似图片的蓝-黄-红渐变
    # colorscale = "Spectral", 
    colorscale = list(c(0, "rgb(68, 115, 197)"),   # 蓝色 (低)
                      c(0.5, "rgb(255, 255, 191)"), # 黄色 (中)
                      c(1, "rgb(236, 43, 36)")),    # 红色 (高)
    # reversescale = TRUE,
    # 配置网格线，使其看起来更像 3D 模型
    contours = list(
      z = list(show = TRUE, usecolormap = TRUE, project = list(z = TRUE))
    ),
    # 调整光照效果，增加立体感
    lighting = list(ambient = 0.6, diffuse = 0.8, specular = 0.2, roughness = 0.5)
) %>%
  layout(
    scene = list(
      xaxis = list(
        title = "Phylum (Time)",
        tickvals = 1:4,
        ticktext = phylum_orders,
        titlefont = list(size = 18, family = "Arial Black")
      ),
      yaxis = list(
        title = "DNB Modules (Ranked)",
        # 如果你想显示具体的 Rank 标签可以自定义 ticktext
        showticklabels = FALSE 
      ),
      zaxis = list(title = "DNB SCORE"),
      camera = list(eye = list(x = 1.9, y = -1.5, z = 1.1)),
      aspectmode = "manual",
      aspectratio = list(x = 1.2, y = 1.2, z = 0.6)
    ),
    title = list(text = "Evolutionary Landscape of Neural Origin Signals", y = 0.95)
  )

# 保存
htmlwidgets::saveWidget(p_3d_final, paste0(out_dir, "/DNB_3D_Landscape_RankOrder.html"))

### Top 10 Gene Modules 静态

In [ ]:
library(ggplot2)

# 1. 确保顺序（承接上文逻辑）
phylum_orders <- c("Porifera", "Placozoa", "Cnidaria", "Bilateria")

ordered_modules <- unique(plot_df$Module_ID)
plot_df$Module_ID <- factor(plot_df$Module_ID, levels = ordered_modules)
plot_df$Names <- factor(plot_df$Names, levels = phylum_orders)

# 2. 绘图：黑线 + 关键节点红点
p_origin_focus <- ggplot(plot_df, aes(x = Names, y = SCORE, group = Module_ID)) +
  # 绘制折线：全部统一为黑色
  geom_line(color = "black", size = 1, alpha = 0.8) +
  # 绘制散点：利用 ifelse 判断，如果是 Placozoa 则为红色，否则为黑色
  geom_point(aes(color = (Names == "Placozoa")), size = 2.5) +
  # 映射颜色：TRUE 为红色，FALSE 为黑色
  scale_color_manual(values = c("TRUE" = "#EC2B24", "FALSE" = "black")) +
  # 分面展示
  facet_wrap(~Module_ID, ncol = 2) + 
  theme_bw(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, face = "bold", color = "black"),
    axis.text.y = element_text(color = "black"),
    strip.background = element_rect(fill = "gray95"),
    strip.text = element_text(face = "bold"),
    legend.position = "none", # 隐藏图例，颜色意图已经很明显
    panel.grid.minor = element_blank()
  ) +
  labs(
    title = "DNB Score Evolution: Focusing on Placozoa Origin",
    subtitle = "Black lines show trends; Red dots highlight the critical transition node",
    x = "Evolutionary Stages", 
    y = "DNB SCORE"
  )

# 3. 保存
ggsave(paste0(out_dir, "/DNB_Facet_Origin_Focus.pdf"), p_origin_focus, width = 10, height = 12)

# 显示
print(p_origin_focus)

### 筛选特定Module的NO-DNBGs

In [ ]:
target_rank <- 1
# 执行“炸开”与导出逻辑
target_rank_mudule <- res_placozoa_only %>%
  # 1. 筛选特定排名 (注意：这里使用局部 rank，如果你想用全局请改为 rank_all)
  filter(rank == target_rank) %>%
  # 2. “炸开”：将逗号分隔的 OG 字符串转为多行
  separate_rows(genes, sep = ",") %>%
  # 3. 清理：去除 OG 编号前后的空格或换行符
  mutate(genes = trimws(genes)) %>%
  # 4. 去重：防止同一个 OG 在同一模块中被多次记录（保险操作）
  distinct(genes, .keep_all = TRUE)

target_rank_og_list <- unique(target_rank_mudule$genes)

# 导出 A：包含统计指标的完整 CSV（方便在 Excel 中查看 SD, SCORE 等）
write.csv(target_rank_mudule, file = paste0(out_dir, "/DNBGs_module_", target_rank, ".csv"), 
            row.names = FALSE)

write.table(target_rank_og_list, file = paste0(out_dir, "/DNBGs_module_", target_rank, "_OGs.txt"), 
            row.names = F, col.names = F, quote = F)

### 筛选统计频次的NO-DNBGs

In [ ]:
library(tidyr)

# 1. 炸开数据并统计
dnb_hub_ogs <- res_placozoa_only %>%
  separate_rows(genes, sep = ",") %>%
  mutate(genes = trimws(genes)) %>%
  group_by(genes) %>%
  summarise(
    Occurrence_Frequency = n(),                  # 该 OG 在多少个模块中出现了
    Max_Module_Score = max(SCORE),               # 所属模块的最高分
    Mean_SD = mean(SD),                          # 模块平均波动水平
    Associated_Resources = paste(resource, collapse = "; "), # 记录都在哪些模块里
    .groups = 'drop'
  ) %>%
  # 优先按出现频率排序，频率相同按 Score 排序
  arrange(desc(Occurrence_Frequency), desc(Max_Module_Score))

# 2. 导出两个文件
# 文件 A: 详细的统计表（包含频率和分数）
write.csv(dnb_hub_ogs, paste0(out_dir, "/Placozoa_DNB_Hub_OGs_Frequency_Stats.csv"), row.names = FALSE)

# 文件 B: 纯净的 DNB-OGs 列表 (用于富集分析或注释匹配)
# 我们只取那些至少在一个高分模块中出现的 OG
final_list <- dnb_hub_ogs$genes
write.table(final_list, paste0(out_dir, "/DNB_OGs_List_Final.txt"), 
            row.names = FALSE, col.names = FALSE, quote = FALSE)

message("策略 B 执行完毕！共找到 ", length(final_list), " 个独特的起源相关 OGs。")